# Monocular Visual Odometry Pipeline

This notebook demonstrates the Monocular Visual Odometry (VO) pipeline. It includes project cloning, dependency installation, test execution, and a synthetic smoke-test to verify the implementation.

## 1. Setup and Project Cloning

In [ ]:
# Clone the repository (skip if already cloned or if you are running this from the repo root)
repo_url = "https://github.com/infi9itea/h-orb-vo.git"
repo_name = "h-orb-vo"

import os
if not os.path.exists(repo_name):
    !git clone {repo_url}
    %cd {repo_name}
else:
    print(f"{repo_name} already exists. Skipping clone.")

## 2. Install Dependencies

In [ ]:
# Install dependencies from requirements.txt
!pip install -r requirements.txt

## 3. Run Verification Tests

Before running the pipeline, let's ensure all components are working correctly by running the project's test suite.

In [ ]:
# Run all tests using pytest
!python -m pytest test_vo.py

## 4. Import Modules

In [ ]:
import sys
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt

# Ensure the current directory is in the path
sys.path.insert(0, os.getcwd())

from main import run_synthetic_test
from vo_pipeline import MonocularVO, VOConfig
from datasets import KITTISequence
from visualization import plot_trajectory_2d

## 5. Synthetic Smoke-Test

The synthetic test generates a box of 3D points and moves a virtual camera through it. This verifies that the feature tracking, pose estimation, and scale recovery (using ground truth) are working correctly.

In [ ]:
# Run the synthetic test and display metrics
ate, rpe = run_synthetic_test()

print("\nSynthetic Test Summary:")
print(f"  ATE RMSE: {ate['rmse']:.4f} m")
print(f"  RPE Trans RMSE: {rpe['trans_rmse']:.4f} m")

## 6. Running on KITTI Dataset

To run on KITTI, you need the KITTI Odometry dataset. If you are on Kaggle, you can add the dataset to your notebook.

**Note:** Update `kitti_root` to the path where your dataset is located (e.g., `/kaggle/input/kitti-odometry-dataset`). For some versions of KITTI on Kaggle, the images might be in `image_2` (color) or `image_3` (color) instead of `image_0` (grayscale).

In [ ]:
# Example Kaggle path (update this with your actual dataset input path)
kitti_root = "/kaggle/input/kitti-odometry-dataset"  
sequence = "00"

if os.path.exists(kitti_root):
    print(f"Loading KITTI sequence {sequence}...")
    # Note: KITTISequence will automatically fall back to other camera indices if image_0 is missing.
    ds = KITTISequence(kitti_root, sequence)
    
    cfg = VOConfig(verbose=True)
    vo = MonocularVO(ds.K, cfg)
    
    # Run for first 100 frames as a demo
    traj = vo.run(iter(ds), max_frames=100)
    
    est_pos = traj.estimated_positions()
    gt_pos = traj.gt_positions_array()
    
    plot_trajectory_2d(est_pos, gt_pos, title=f"KITTI {sequence} (First 100 frames)")
else:
    print(f"KITTI dataset not found at {kitti_root}. Skipping demo.")